<a href="https://colab.research.google.com/github/sanchi23002/pytorch_projects_/blob/main/training_pipeline_using_dataset_and_dataloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [30]:
from torch.utils.data import Dataset , DataLoader

In [31]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [32]:
df.drop(columns = ['id','Unnamed: 32'], inplace = True)
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [33]:
df.shape

(569, 31)

In [34]:
x_train, x_test, y_train, y_test = train_test_split(df.iloc[:,1:],df.loc[:,'diagnosis'], test_size=0.2)
print(x_train.shape)
print(y_train.shape)

(455, 30)
(455,)


**Scaling**

In [35]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

**Label Encoding**

In [36]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

**Numpy array to pytorch tensor**

In [47]:
x_train_tensor = torch.from_numpy(x_train.astype(np.float32))
x_test_tensor = torch.from_numpy(x_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

**Dataset class use**

In [48]:
class customdataset(Dataset):

  def __init__(self,features,labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return len(self.features)

  def __getitem__(self,idx):
    return self.features[idx], self.labels[idx]

**make separate object dataset for training and testing**

In [49]:
train_dataset = customdataset(x_train_tensor, y_train_tensor)
test_dataset = customdataset(x_test_tensor,y_test_tensor)

**using dataloader**

In [50]:
train_loader = DataLoader(train_dataset, batch_size = 16, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 16, shuffle = True)

**Defining the model**

In [41]:
import torch.nn as nn

In [51]:
class my_nn(nn.Module):

  def __init__(self,features):
    super().__init__()
    self.seq = nn.Sequential(
        nn.Linear(features,16),
        nn.ReLU(),
        nn.Linear(16,8),
        nn.ReLU(),
        nn.Linear(8,4),
        nn.ReLU(),
        nn.Linear(4,1),
        nn.Sigmoid()
    )

  def forward(self,features):
    return self.seq(features)

**Important parameters**

In [54]:
learning_rate = 0.01
epochs = 100

In [52]:
model = my_nn(x_train_tensor.shape[1])

loss_function = nn.BCELoss()

optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

**Training Pipeline**

In [55]:
for epoch in range(epochs):
  for batch_features, batch_lebels in train_loader:
    y_pred = model(batch_features)
    loss = loss_function(y_pred, batch_lebels.view(-1,1))## match the shape , -1 find the row number and 1 tell it that column is 1
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print("epoch :", epoch, "loss :", loss.item())


epoch : 0 loss : 0.0020344476215541363
epoch : 0 loss : 0.004305665381252766
epoch : 0 loss : 0.003611672902479768
epoch : 0 loss : 0.001565432408824563
epoch : 0 loss : 0.0004258797853253782
epoch : 0 loss : 0.0001539730146760121
epoch : 0 loss : 0.00042611159733496606
epoch : 0 loss : 0.0010986183770000935
epoch : 0 loss : 0.0010586305288597941
epoch : 0 loss : 0.002250030403956771
epoch : 0 loss : 0.010390213690698147
epoch : 0 loss : 0.009120509959757328
epoch : 0 loss : 0.0035444912500679493
epoch : 0 loss : 0.0005000949022360146
epoch : 0 loss : 0.000555310514755547
epoch : 0 loss : 0.0004529035068117082
epoch : 0 loss : 0.00033939906279556453
epoch : 0 loss : 0.0001675371458986774
epoch : 0 loss : 0.0002856212668120861
epoch : 0 loss : 0.00017618166748434305
epoch : 0 loss : 0.001134214224293828
epoch : 0 loss : 0.000760396127589047
epoch : 0 loss : 0.0006441941950470209
epoch : 0 loss : 0.0015243809903040528
epoch : 0 loss : 0.0048477849923074245
epoch : 0 loss : 0.000215462336

In [58]:
model.eval() ## sets the model to evaluation mode,disables dropout and batch normalization, to ensure consistent predictions.
accuracy_list = [] ## An empty list is initialized to store the accuracy calculated for each batch of test data.
with torch.no_grad():## disables gradient calculation. During evaluation, you don't need to compute gradients .
  for batch_features, batch_lebels in test_loader:
    y_pred = model(batch_features)
    y_pred = (y_pred>0.6).float()
## accuracy of batch calculation
    batch_accuracy = (y_pred.view(-1)==batch_lebels).float().mean().item()
    print("accuracy : ", batch_accuracy)
    accuracy_list.append(batch_accuracy)## The batch accuracy is added to the accuracy_list.

overall_accuracy = sum(accuracy_list)/len(accuracy_list)
print("length :  ", len(accuracy_list))
print("overall accuracy :  ", overall_accuracy)

accuracy :  1.0
accuracy :  1.0
accuracy :  0.9375
accuracy :  1.0
accuracy :  1.0
accuracy :  1.0
accuracy :  1.0
accuracy :  1.0
length :   8
overall accuracy :   0.9921875


In [59]:
print('\n--- Alternative Accuracy Calculation ---')

# Assuming y_pred has a shape like (batch_size, 1)
# and batch_lebels has a shape like (batch_size)

# Unsqueeze batch_lebels to match y_pred's shape (batch_size, 1)
batch_accuracy_alt = (y_pred == batch_lebels.unsqueeze(1)).float().mean().item()
##  In PyTorch, unsqueeze(dim) adds a new dimension of size one at the specified position dim.
##   you are transforming its shape from (batch_size,) to (batch_size, 1)
print("Alternative batch accuracy : ", batch_accuracy_alt)



--- Alternative Accuracy Calculation ---
Alternative batch accuracy :  1.0
